In [2]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR1TOT_D.xpt"
csv_file = DATA_DIR / "DR1TOT_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 DR1TOT_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

if len(df) != 9950:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )

if len(df.columns) != 160:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )

if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


REQUIRED_COLUMNS = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DR1DRSTZ",
    "DR1TNUMF",
    "DR1TKCAL",
    "DR1TPROT",
    "DR1TP204",
    "DR1TP226",
    "DR1_320Z",
    "DR1_330Z",
    "DR1BWATZ",
    "DR1TWS"
]


missing = [
    col
    for col in REQUIRED_COLUMNS
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Expected columns missing: {missing}"
    )


print(
    "PASS: DR1TOT_D structure = "
    "9,950 rows x 160 columns."
)


# ============================================================
# 4. DEFINE MEASUREMENT VARIABLES
# ============================================================

#
# Dietary measurements MUST remain AS IS.
#
# Nutrient block:
# DR1TKCAL through DR1TP226
#
# Plus:
# WTDRD1
# WTDR2D
# DR1_320Z
# DR1_330Z
# DR1BWATZ
#

nutrient_start = df.columns.get_loc(
    "DR1TKCAL"
)

nutrient_end = df.columns.get_loc(
    "DR1TP226"
)


NUTRIENT_COLUMNS = (
    df.columns[
        nutrient_start:
        nutrient_end + 1
    ]
    .tolist()
)


MEASUREMENT_COLUMNS = [
    "WTDRD1",
    "WTDR2D",
] + NUTRIENT_COLUMNS + [
    "DR1_320Z",
    "DR1_330Z",
    "DR1BWATZ"
]


print(
    "Dietary measurement fields:",
    len(MEASUREMENT_COLUMNS)
)


# ============================================================
# 5. KNOWN PANDAS/XPORT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


true_zero_count = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{true_zero_count:,}"
)

print(
    "Tiny artifact occurrences:",
    f"{int(tiny_counts.sum()):,}"
)

print(
    "Variables containing tiny values:",
    int((tiny_counts > 0).sum())
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. EXPECTED TINY COUNTS FROM THIS XPT
# ============================================================

EXPECTED_TINY_COUNTS = {

    "DR1BWATZ": 6325,

    "DR1TACAR": 1058,
    "DR1TALCO": 7815,
    "DR1TATOA": 8403,
    "DR1TATOC": 2,

    "DR1TB12A": 5828,
    "DR1TBCAR": 128,
    "DR1TCAFF": 2430,
    "DR1TCARB": 1,
    "DR1TCHL": 2,
    "DR1TCHOL": 43,
    "DR1TCRYP": 934,

    "DR1TFA": 155,
    "DR1TFDFE": 2,
    "DR1TFF": 84,
    "DR1TFIBE": 97,
    "DR1TFOLA": 2,

    "DR1TIRON": 1,
    "DR1TKCAL": 1,
    "DR1TLYCO": 2407,
    "DR1TLZ": 112,

    "DR1TM161": 26,
    "DR1TM181": 2,
    "DR1TM201": 99,
    "DR1TM221": 3329,
    "DR1TMFAT": 2,

    "DR1TNIAC": 1,

    "DR1TP182": 2,
    "DR1TP183": 3,
    "DR1TP184": 6212,
    "DR1TP204": 351,
    "DR1TP205": 2835,
    "DR1TP225": 4171,
    "DR1TP226": 2261,
    "DR1TPFAT": 2,
    "DR1TPHOS": 1,
    "DR1TPOTA": 1,
    "DR1TPROT": 2,

    "DR1TRET": 61,

    "DR1TS040": 585,
    "DR1TS060": 497,
    "DR1TS080": 310,
    "DR1TS100": 120,
    "DR1TS120": 28,
    "DR1TS140": 4,
    "DR1TS160": 2,
    "DR1TS180": 2,
    "DR1TSELE": 2,
    "DR1TSFAT": 2,
    "DR1TSUGR": 1,

    "DR1TTFAT": 2,
    "DR1TTHEO": 4648,

    "DR1TVARA": 16,
    "DR1TVB1": 1,
    "DR1TVB12": 16,
    "DR1TVB2": 2,
    "DR1TVB6": 1,
    "DR1TVC": 23,
    "DR1TVK": 2,

    "DR1_320Z": 2633,
    "DR1_330Z": 5046
}


ZERO_ALLOWED_COLUMNS = list(
    EXPECTED_TINY_COUNTS.keys()
)


# ============================================================
# 7. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

approved_tiny_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected = (
    actual_tiny_columns
    -
    approved_tiny_columns
)


if unexpected:

    raise ValueError(
        "STOP: Tiny values found in unexpected "
        "DR1TOT_D variables:\n"
        + str(sorted(unexpected))
    )


missing_expected = (
    approved_tiny_columns
    -
    actual_tiny_columns
)


if missing_expected:

    raise ValueError(
        "Expected tiny-value variables missing:\n"
        + str(sorted(missing_expected))
    )


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_count = int(
        tiny_counts[col]
    )

    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} "
            f"tiny values, found {actual_count}."
        )


if int(tiny_counts.sum()) != 69134:

    raise ValueError(
        "Unexpected total tiny-value count."
    )


print(
    "\nPASS: All 69,134 tiny values occur only "
    "in validated zero-permitted dietary fields."
)


# ============================================================
# 8. RESTORE VALIDATED ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )

    count = int(
        mask.sum()
    )

    df.loc[
        mask,
        col
    ] = 0.0

    print(
        f"{col:<12}: "
        f"{count:,} tiny -> 0"
    )


# ============================================================
# 9. VERIFY RESTORED ZEROS
# ============================================================

print("\n--- RESTORED ZERO CHECK ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero = int(
        df[col]
        .eq(0)
        .sum()
    )

    if actual_zero != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count} zeros, "
            f"found {actual_zero}."
        )


print(
    "PASS: All 69,134 validated zeros restored."
)


# ============================================================
# 10. CODE / COUNT FIELDS
# ============================================================

#
# Everything numeric that is NOT:
#
# SEQN
# or a dietary measurement
#
# should be whole-number coded/count data.
#

CODE_COLUMNS = [
    col
    for col in numeric_cols
    if col not in (
        ["SEQN"]
        + MEASUREMENT_COLUMNS
    )
]


print(
    "Whole-number code/count fields:",
    len(CODE_COLUMNS)
)


# ============================================================
# 11. VERIFY CODE FIELDS HAVE NO REAL FRACTIONS
# ============================================================

print("\n--- CODE FIELD FRACTION CHECK ---")


fraction_failures = []


for col in ["SEQN"] + CODE_COLUMNS:

    values = (
        df[col]
        .dropna()
    )

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        fraction_failures.append(
            col
        )

        print(
            f"\nUnexpected fractional values in {col}:"
        )

        print(
            values[
                fractional
            ].head(20)
        )


if fraction_failures:

    raise ValueError(
        "STOP: Real decimal values detected in "
        "code/count fields:\n"
        + str(fraction_failures)
    )


print(
    "PASS: All identifier/code/count fields "
    "contain whole-number values."
)


# ============================================================
# 12. REPORT REAL DIETARY DECIMALS
# ============================================================

print("\n--- DIETARY DECIMAL CHECK ---")


fractional_measurements = []


for col in MEASUREMENT_COLUMNS:

    values = (
        df[col]
        .dropna()
    )

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional.any():

        fractional_measurements.append(
            col
        )


print(
    "Measurement fields containing genuine "
    "fractions:",
    len(fractional_measurements)
)


print(
    fractional_measurements
)


#
# In this attached XPT there are 49 measurement
# variables containing genuine fractional values.
#

if len(fractional_measurements) != 49:

    raise ValueError(
        "Unexpected number of fractional "
        "measurement variables."
    )


print(
    "PASS: Genuine dietary decimals identified "
    "and will remain untouched."
)


# ============================================================
# 13. CLEAN SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractions."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31127
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 14. FORMAT ONLY CODE / COUNT FIELDS
# ============================================================

#
# NO .round()
#
# Dietary measurements are NOT included here.
#

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Code/count fields formatted "
    "without unnecessary .0."
)


# ============================================================
# 15. CDC CHECK - DIETARY RECALL STATUS
# ============================================================

EXPECTED_DR1DRSTZ = {
    1: 9169,
    2: 184,
    4: 180,
    5: 417
}


actual = (
    df["DR1DRSTZ"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_DR1DRSTZ:

    raise ValueError(
        "DR1DRSTZ differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "\nPASS: DR1DRSTZ matches CDC."
)


# ============================================================
# 16. CDC CHECK - NUMBER OF FOODS
# ============================================================

s = df["DR1TNUMF"]


if not (
    int(
        s.between(
            2,
            57
        ).sum()
    )
    ==
    9349

    and

    int(
        s.isna()
        .sum()
    )
    ==
    601

    and

    int(
        s.eq(0)
        .sum()
    )
    ==
    0
):

    raise ValueError(
        "DR1TNUMF differs from CDC."
    )


print(
    "PASS: DR1TNUMF = 2-57, "
    "9,349 valid, 601 missing."
)


# ============================================================
# 17. SAMPLE WEIGHT CHECK
# ============================================================

if not (
    int(
        df["WTDRD1"]
        .notna()
        .sum()
    )
    ==
    9349

    and

    int(
        df["WTDRD1"]
        .isna()
        .sum()
    )
    ==
    601
):

    raise ValueError(
        "WTDRD1 count differs from CDC."
    )


if not (
    int(
        df["WTDR2D"]
        .notna()
        .sum()
    )
    ==
    8429

    and

    int(
        df["WTDR2D"]
        .isna()
        .sum()
    )
    ==
    1521
):

    raise ValueError(
        "WTDR2D count differs."
    )


print(
    "PASS: Dietary sample weight counts validated."
)


# ============================================================
# 18. NUTRIENT COMPLETENESS CHECK
# ============================================================

#
# Every nutrient total from DR1TKCAL through DR1TP226:
#
# valid   = 9,169
# missing = 781
#

nutrient_failures = []


for col in NUTRIENT_COLUMNS:

    valid = int(
        df[col]
        .notna()
        .sum()
    )

    missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        valid != 9169
        or
        missing != 781
    ):

        nutrient_failures.append(
            (
                col,
                valid,
                missing
            )
        )


if nutrient_failures:

    raise ValueError(
        "Nutrient completeness differs:\n"
        + str(nutrient_failures)
    )


print(
    "PASS: All 64 nutrient fields have "
    "9,169 valid and 781 missing."
)


# ============================================================
# 19. WATER TOTAL CHECKS
# ============================================================

WATER_CHECKS = {

    "DR1_320Z": {
        "min": 0,
        "max": 10000,
        "valid": 9349,
        "missing": 601,
        "zeros": 2633
    },

    "DR1_330Z": {
        "min": 0,
        "max": 7584,
        "valid": 9349,
        "missing": 601,
        "zeros": 5046
    },

    "DR1BWATZ": {
        "min": 0,
        "max": 10000,
        "valid": 9349,
        "missing": 601,
        "zeros": 6325
    }
}


print("\n--- WATER TOTAL CHECKS ---")


for col, spec in WATER_CHECKS.items():

    s = df[col]


    if not (
        int(s.notna().sum())
        ==
        spec["valid"]

        and

        int(s.isna().sum())
        ==
        spec["missing"]

        and

        np.isclose(
            s.min(),
            spec["min"]
        )

        and

        np.isclose(
            s.max(),
            spec["max"]
        )

        and

        int(s.eq(0).sum())
        ==
        spec["zeros"]
    ):

        raise ValueError(
            f"{col} differs from CDC/source."
        )


    print(
        f"PASS: {col} "
        f"range={s.min()}-{s.max()} "
        f"zeros={int(s.eq(0).sum()):,}"
    )


# ============================================================
# 20. BUILD EXPECTED SOURCE
# ============================================================

#
# Expected source =
#
# original XPT
# +
# ONLY the 61 validated tiny -> 0 corrections.
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 21. EXACT VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    if (
        col == "SEQN"
        or
        col in CODE_COLUMNS
    ):

        current_values = (
            df[col]
            .astype("Float64")
            .to_numpy(
                dtype=float,
                na_value=np.nan
            )
        )

    else:

        current_values = (
            df[col]
            .to_numpy()
        )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(failures)
    )


print(
    "PASS: All 160 variables preserved exactly "
    "after validated zero restoration."
)


# ============================================================
# 22. FINAL TINY CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nRemaining tiny values:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


print(
    "PASS: No tiny-value artifact remains."
)


# ============================================================
# 23. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format
# NO rounding
# NO recalculation
# NO decimal trimming
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 24. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 25. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 160 CSV columns numerically "
    "match expected XPT values."
)


# ============================================================
# 26. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            csv_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 27. RAW CSV FIDELITY
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 28. SEQN / CODE .0 CHECK
# ============================================================

bad_code_dot_zero = 0


for row in raw_rows:

    for col in ["SEQN"] + CODE_COLUMNS:

        value = row[col]

        if value.endswith(".0"):

            bad_code_dot_zero += 1


print(
    "\nSEQN/code fields ending '.0':",
    bad_code_dot_zero
)


if bad_code_dot_zero != 0:

    raise ValueError(
        "Unnecessary .0 remains in "
        "identifier/code fields."
    )


print(
    "PASS: SEQN/code/count fields have "
    "no unnecessary .0."
)


# ============================================================
# 29. RAW DIETARY MEASUREMENT FIDELITY
# ============================================================

#
# Compare CSV token directly with:
#
# str(float(expected XPT value))
#
# Therefore values such as:
#
# 0.028999999999999998
#
# remain exactly that way if that is how pandas
# decodes the XPT.
#

measurement_mismatches = {
    col: 0
    for col in MEASUREMENT_COLUMNS
}


for i, row in enumerate(raw_rows):

    for col in MEASUREMENT_COLUMNS:

        xpt_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(xpt_value)
            else str(float(xpt_value))
        )


        if row[col] != expected_text:

            measurement_mismatches[col] += 1


bad_measurements = {
    col: count
    for col, count
    in measurement_mismatches.items()
    if count > 0
}


print(
    "Dietary measurement raw-text mismatches:",
    bad_measurements
)


if bad_measurements:

    raise ValueError(
        "Dietary measurement formatting changed."
    )


print(
    "PASS: All dietary measurements/weights "
    "appear AS IS in CSV."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("DR1TOT_D CONVERSION COMPLETE")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))
print("Validated tiny -> zero:", "69,134")
print("Tiny-value variables:", 61)
print("Measurement variables:", len(MEASUREMENT_COLUMNS))
print(
    "Measurement fields containing real fractions:",
    len(fractional_measurements)
)
print("XPT -> CSV differences: 0 expected")
print("Missing differences: 0 expected")
print("Code/SEQN .0: 0 expected")

NHANES 2005-2006 DR1TOT_D XPT -> CSV
Rows: 9,950
Columns: 160
PASS: DR1TOT_D structure = 9,950 rows x 160 columns.
Dietary measurement fields: 69

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 69,134
Variables containing tiny values: 61

Tiny values by variable:
DR1TATOA    8403
DR1TALCO    7815
DR1BWATZ    6325
DR1TP184    6212
DR1TB12A    5828
DR1_330Z    5046
DR1TTHEO    4648
DR1TP225    4171
DR1TM221    3329
DR1TP205    2835
DR1_320Z    2633
DR1TCAFF    2430
DR1TLYCO    2407
DR1TP226    2261
DR1TACAR    1058
DR1TCRYP     934
DR1TS040     585
DR1TS060     497
DR1TP204     351
DR1TS080     310
DR1TFA       155
DR1TBCAR     128
DR1TS100     120
DR1TLZ       112
DR1TM201      99
DR1TFIBE      97
DR1TFF        84
DR1TRET       61
DR1TCHOL      43
DR1TS120      28
DR1TM161      26
DR1TVC        23
DR1TVARA      16
DR1TVB12      16
DR1TS140       4
DR1TP183       3
DR1TCHL        2
DR1TPFAT       2
DR1TMFAT       2
DR1TSFAT       2
DR1TTFAT       2
DR1TA